In [ ]:
import sqlite3
from sklearn.linear_model import LogisticRegression
connect = sqlite3.connect("../db/pl_data.db")
cursor = connect.cursor()

In [1]:
import pandas as pd
import sqlite3

connect = sqlite3.connect("../db/pl_data.db")
matches = pd.read_sql_query("SELECT * FROM matches", connect)

In [3]:
query = "SELECT DISTINCT team_id, season_year FROM matches"
unique_pairs = pd.read_sql_query(query, connect)

DatabaseError: Execution failed on sql 'SELECT DISTINCT team_id, season_year FROM matches': no such column: team_id

In [ ]:
# confirming total games
cursor.execute("SELECT season_year, COUNT(*) FROM matches GROUP BY season_year")
cursor.fetchall()

In [ ]:
# get teams for CSV matching
cursor.execute("SELECT team_id, name FROM teams")
cursor.fetchall()

In [28]:
import pandas as pd
match_features = pd.read_csv("../data/processed/match_features.csv")

feature_cols = [
    "matchday",
    "implied_prob_home", "implied_prob_draw", "implied_prob_away",

    # home side rolling form
    "rolling_points_5_home",
    "rolling_goals_for_5_home", "rolling_goals_against_5_home",
    "rolling_shots_for_5_home", "rolling_shots_against_5_home",
    "rolling_shots_target_for_5_home", "rolling_shots_target_against_5_home",
    "rolling_corners_for_5_home", "rolling_corners_against_5_home",
    "rolling_yellow_for_5_home", "rolling_yellow_against_5_home",
    "rolling_red_for_5_home", "rolling_red_against_5_home",
    "rolling_points_5_by_venue_home",
    "rolling_goals_for_5_by_venue_home", "rolling_goals_against_5_by_venue_home",
    "rest_days_home", "long_layoff_home",
    "h2h_points_avg_3_home",

    # away side rolling form
    "rolling_points_5_away",
    "rolling_goals_for_5_away", "rolling_goals_against_5_away",
    "rolling_shots_for_5_away", "rolling_shots_against_5_away",
    "rolling_shots_target_for_5_away", "rolling_shots_target_against_5_away",
    "rolling_corners_for_5_away", "rolling_corners_against_5_away",
    "rolling_yellow_for_5_away", "rolling_yellow_against_5_away",
    "rolling_red_for_5_away", "rolling_red_against_5_away",
    "rolling_points_5_by_venue_away",
    "rolling_goals_for_5_by_venue_away", "rolling_goals_against_5_by_venue_away",
    "rest_days_away", "long_layoff_away",
    "h2h_points_avg_3_away",
]

X = match_features[feature_cols]
y = match_features["winner"]

In [29]:
X.shape

(3038, 42)

In [ ]:
y.value_counts()

In [ ]:
match_features[match_features["season_year"].isin([2018, 2019, 2020, 2021, 2022, 2023, 2024])]


In [32]:
train = match_features[match_features["season_year"].isin([2018, 2019, 2020, 2021, 2022, 2023, 2024])]
test = match_features[match_features["season_year"] == 2025]

X_train = train[feature_cols]
y_train = train["winner"]

X_test = test[feature_cols]
y_test = test["winner"]

In [ ]:
X_train.isna().sum()

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

In [1]:
import pandas as pd
fixtures = pd.read_csv("../data/raw/fixtures_2026-27.csv")
print(fixtures.shape)

home_counts = fixtures["home_team_id"].value_counts()
away_counts = fixtures["away_team_id"].value_counts()
total_per_team = home_counts.add(away_counts, fill_value=0)
print(total_per_team.sort_values())

(380, 5)
home_team_id
57      38
58      38
61      38
62      38
63      38
64      38
65      38
66      38
67      38
73      38
351     38
354     38
397     38
402     38
1044    38
9005    38
9008    38
9009    38
9010    38
9011    38
Name: count, dtype: int64


In [3]:
import sqlite3
connect = sqlite3.connect("../db/pl_data.db")
cursor = connect.cursor()
cursor.execute("SELECT COUNT(*) FROM teams")
print(cursor.fetchall())  # should show 32

[(32,)]


In [2]:
import json
import pandas as pd

with open("../data/processed/simulation_results_50k_final.json", "r") as f:
    all_results = json.load(f)

print(len(all_results))
results_df = pd.DataFrame(all_results)
print(results_df.shape)

50000
(50000, 20)


In [3]:
top4_counts = {team_id: 0 for team_id in results_df.columns}

for _, row in results_df.iterrows():
    top4_teams = row.sort_values(ascending=False).head(4).index
    for team_id in top4_teams:
        top4_counts[team_id] += 1

top4_odds = pd.Series(top4_counts).sort_values(ascending=False) / len(results_df) * 100
print(top4_odds)

57      28.012
66      24.334
1044    24.008
73      23.066
58      22.222
9009    22.212
63      21.636
9005    20.772
9011    19.560
65      19.158
67      19.044
354     18.726
397     18.612
9010    18.156
64      18.020
402     17.922
351     17.126
61      16.896
9008    15.376
62      15.142
dtype: float64


In [4]:
relegation_counts = {team_id: 0 for team_id in results_df.columns}

for _, row in results_df.iterrows():
    bottom3_teams = row.sort_values(ascending=True).head(3).index
    for team_id in bottom3_teams:
        relegation_counts[team_id] += 1

relegation_odds = pd.Series(relegation_counts).sort_values(ascending=False) / len(results_df) * 100
print(relegation_odds)

9010    21.022
9011    19.974
62      18.980
9008    18.532
402     17.164
61      17.048
64      16.284
351     15.854
397     15.814
354     15.522
65      14.422
67      14.106
58      13.548
9005    13.456
63      13.098
1044    11.938
9009    11.838
73      11.222
66      10.536
57       9.642
dtype: float64


In [25]:
import random

random_index = random.randint(0, len(results_df) - 1)
one_season = results_df.iloc[random_index].sort_values(ascending=False)
print(f"Simulation #{random_index}")
print(one_season)

Simulation #31760
9011    80
1044    71
58      63
65      60
9009    60
351     59
9008    58
61      57
402     56
62      55
9005    53
57      52
397     52
64      47
9010    47
66      46
67      43
354     37
63      29
73      28
Name: 31760, dtype: int64


In [19]:
results_df["65"].describe()

count    20000.000000
mean        52.990950
std         12.028487
min         14.000000
25%         45.000000
50%         53.000000
75%         61.000000
max        102.000000
Name: 65, dtype: float64

In [44]:
import json
with open("../data/processed/simulation_results.json", "r") as f:
    data = json.load(f)
print(len(data))

2000


In [14]:
team_names = {
    57: "Arsenal", 58: "Aston Villa", 1044: "AFC Bournemouth", 402: "Brentford",
    397: "Brighton & Hove Albion", 61: "Chelsea", 9010: "Coventry City",
    354: "Crystal Palace", 62: "Everton", 63: "Fulham", 9011: "Hull City",
    9008: "Ipswich Town", 9005: "Leeds United", 64: "Liverpool", 65: "Manchester City",
    66: "Manchester United", 67: "Newcastle United", 351: "Nottingham Forest",
    9009: "Sunderland", 73: "Tottenham Hotspur",
}

summary = pd.DataFrame({
    "team": [team_names[int(t)] for t in results_df.columns],
    "avg_points": results_df.mean().values,
    "title_pct": title_odds.reindex(results_df.columns).values,
    "top4_pct": top4_odds.reindex(results_df.columns).values,
    "relegation_pct": relegation_odds.reindex(results_df.columns).values,
})

summary = summary.sort_values("top4_pct", ascending=False).reset_index(drop=True)
print(summary)

NameError: name 'title_odds' is not defined

In [3]:
import json
import pandas as pd

with open("../data/processed/simulation_results_50k_final.json", "r") as f:
    all_results = json.load(f)
print(len(all_results))

summary = pd.read_csv("../data/processed/season_predictions_summary.csv")
print(summary.shape)
print(summary.head())

50000
(20, 5)
                team  avg_points  title_pct  top4_pct  relegation_pct
0            Arsenal        55.9        8.0      28.0             9.6
1  Manchester United        55.2        6.4      24.3            10.5
2    AFC Bournemouth        54.4        6.4      24.0            11.9
3  Tottenham Hotspur        54.7        5.8      23.1            11.2
4        Aston Villa        53.5        5.8      22.2            13.5
